# Тахи · сегментация ушей, ног и корпуса

Проверка DINOv3 запускается без данных заповедника. Для обучения включите GPU и Internet, добавьте приватный датасет с `manifest.json`, `photos/` и `labels/` и укажите его путь в ячейке настроек.

Ручные контуры имеют формат `{"source":"manual","polygons":{"ears":[[[x,y],…]],"legs":[…],"body":[…]}}`. Координаты от 0 до 1. Размечайте видимые части одной целевой лошади на кадре. На закрытой контрольной части разделяем кадры по датам. Псевдоразметка оценивается по той же ручной контрольной части.


In [ ]:
%pip -q install 'timm>=1.0.20,<2' 'opencv-python-headless>=4.8,<5'

In [ ]:
from pathlib import Path
import os

DATASET = None  # Например, Path('/kaggle/input/datasets/USER/takhi-labels')
BACKBONE_WEIGHTS = None  # Путь к локальным весам timm при отключённом Internet
RUN_PSEUDO = False
WORK = Path("/kaggle/working")
os.environ["HF_HOME"] = "/tmp/takhi-hf"


In [ ]:
program = '"""Train or apply a horse-part mask head over frozen DINOv3 features."""\nimport argparse\nfrom collections import defaultdict\nimport json\nimport os\nfrom pathlib import Path\nimport random\n\nimport numpy as np\nfrom PIL import Image, ImageDraw, ImageOps\nimport timm\nimport torch\nfrom torch import nn\nfrom torch.nn import functional as F\nfrom torchvision.transforms import Compose, Normalize, Resize, ToTensor\n\nPARTS = ("body", "legs", "ears")\nMODEL = "vit_small_patch16_dinov3.lvd1689m"\nSIZE = 448\nPREPROCESS = Compose([\n    Resize((SIZE, SIZE), antialias=True),\n    ToTensor(),\n    Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225)),\n])\n\n\ndef device():\n    if torch.cuda.is_available():\n        return torch.device("cuda")\n    if torch.backends.mps.is_available() and not os.environ.get("TAKHI_CPU"):\n        return torch.device("mps")\n    return torch.device("cpu")\n\n\ndef encoder(target, weights=None):\n    model = timm.create_model(MODEL, pretrained=weights is None, img_size=SIZE,\n                              checkpoint_path=str(weights) if weights else "")\n    return model.eval().requires_grad_(False).to(target)\n\n\n@torch.no_grad()\ndef features(model, images):\n    tokens = model.forward_features(images)[:, model.num_prefix_tokens:]\n    side = SIZE // 16\n    return tokens.transpose(1, 2).reshape(len(images), model.num_features, side, side)\n\n\ndef image_tensor(path):\n    with Image.open(path) as image:\n        return PREPROCESS(ImageOps.exif_transpose(image).convert("RGB"))\n\n\ndef mask_tensor(polygons):\n    mask = Image.new("L", (SIZE, SIZE))\n    draw = ImageDraw.Draw(mask)\n    for index, part in enumerate(PARTS, 1):\n        for polygon in polygons.get(part, []):\n            draw.polygon([(round(x * (SIZE - 1)), round(y * (SIZE - 1))) for x, y in polygon], fill=index)\n    return torch.from_numpy(np.array(mask, dtype=np.int64))\n\n\ndef head(channels):\n    return nn.Sequential(nn.Conv2d(channels, 96, 1), nn.GELU(),\n                         nn.Conv2d(96, len(PARTS) + 1, 3, padding=1))\n\n\ndef logits(model, patches):\n    return F.interpolate(model(patches), (SIZE, SIZE), mode="bilinear", align_corners=False)\n\n\ndef data_rows(root):\n    manifest = json.loads((root / "manifest.json").read_text())\n    for horse, photos in manifest["horses"].items():\n        for photo in photos:\n            yield horse, photo["capturedOn"], photo["id"]\n\n\ndef model_features(model, target, root, keys):\n    for index in range(0, len(keys), 4):\n        batch = keys[index:index + 4]\n        images = torch.stack([image_tensor(root / "photos" / f"{key}.jpg") for key in batch])\n        yield from features(model, images.to(target)).cpu()\n\n\ndef train(args):\n    labels = {}\n    for horse, day, key in data_rows(args.dataset):\n        annotation = args.dataset / "labels" / f"{key}.json"\n        if annotation.is_file():\n            record = json.loads(annotation.read_text())\n            if record.get("source") == "manual" and record.get("polygons"):\n                labels[key] = (horse, day, record["polygons"])\n    groups = defaultdict(list)\n    for key, (_, day, _) in labels.items():\n        groups[day].append(key)\n    days = sorted(groups)\n    if len(labels) < 10 or len(days) < 3:\n        raise SystemExit("Need at least 10 manual masks covering at least three shooting dates.")\n    validation_days = set(days[-max(1, len(days) // 5):])\n    training = [key for day in days if day not in validation_days for key in groups[day]]\n    validation = [key for day in days if day in validation_days for key in groups[day]]\n    if not all(any(labels[key][2].get(part) for key in training) for part in PARTS):\n        raise SystemExit("Training annotations must cover ears, legs and body.")\n    torch.manual_seed(19)\n    target = device()\n    backbone = encoder(target, args.weights)\n    keys = training + validation\n    vectors = dict(zip(keys, model_features(backbone, target, args.dataset, keys), strict=True))\n    masks = {key: mask_tensor(labels[key][2]) for key in keys}\n    if args.pseudo:\n        pseudo_path = args.dataset / "predictions.json"\n        predictions = json.loads(pseudo_path.read_text())\n        eligible = {key for _, day, key in data_rows(args.dataset) if day < min(validation_days)}\n        pseudo = [key for key, record in predictions.items() if key in eligible and key not in labels\n                  and all(record["partConfidence"].get(part, 0) >= args.pseudo_threshold for part in PARTS)]\n        pseudo = pseudo[:len(training)]\n        vectors.update(zip(pseudo, model_features(backbone, target, args.dataset, pseudo), strict=True))\n        masks.update({key: mask_tensor(predictions[key]["polygons"]) for key in pseudo})\n    else:\n        pseudo = []\n\n    decoder = head(backbone.num_features).to(target)\n    optimizer = torch.optim.AdamW(decoder.parameters(), lr=0.002)\n    frequencies = torch.bincount(torch.cat([masks[key].flatten() for key in training]), minlength=4)\n    weights = frequencies.float().clamp_min(1).rsqrt()\n    weights = (weights / weights.mean()).to(target)\n    mixed = training + pseudo\n    rng = random.Random(19)\n    for _ in range(args.epochs):\n        rng.shuffle(mixed)\n        for key in mixed:\n            prediction = logits(decoder, vectors[key].unsqueeze(0).to(target))\n            loss = F.cross_entropy(prediction, masks[key].unsqueeze(0).to(target), weight=weights)\n            optimizer.zero_grad()\n            (loss * (1 if key in labels else 0.35)).backward()\n            optimizer.step()\n\n    confusion = torch.zeros(16, dtype=torch.int64)\n    decoder.eval()\n    with torch.no_grad():\n        for key in validation:\n            prediction = logits(decoder, vectors[key].unsqueeze(0).to(target)).argmax(1)[0].cpu()\n            confusion += torch.bincount((masks[key] * 4 + prediction).flatten(), minlength=16)\n    confusion = confusion.reshape(4, 4)\n    union = confusion.sum(0) + confusion.sum(1) - confusion.diag()\n    iou = {part: round((confusion[i, i] / union[i]).item(), 4) if union[i] else None\n           for i, part in enumerate(("background", *PARTS))}\n    args.output.parent.mkdir(parents=True, exist_ok=True)\n    torch.save({"backbone": MODEL, "size": SIZE, "state_dict": decoder.cpu().state_dict()}, args.output)\n    report = {"trainImages": len(training), "manualValidationImages": len(validation), "pseudoImages": len(pseudo),\n              "validationDates": sorted(validation_days), "iou": iou, "device": str(target)}\n    args.output.with_suffix(".json").write_text(json.dumps(report, indent=2) + "\\n")\n    print(json.dumps(report))\n\n\n@torch.inference_mode()\ndef predict(args):\n    import cv2\n\n    target = device()\n    backbone = encoder(target, args.weights)\n    checkpoint = torch.load(args.checkpoint, map_location="cpu", weights_only=True)\n    if (checkpoint["backbone"], checkpoint["size"]) != (MODEL, SIZE):\n        raise ValueError("Incompatible horse part checkpoint")\n    decoder = head(backbone.num_features).to(target)\n    decoder.load_state_dict(checkpoint["state_dict"])\n    decoder.eval()\n\n    def result(patches):\n        distribution = logits(decoder, patches.unsqueeze(0).to(target)).softmax(1)[0].cpu().numpy()\n        classes = distribution.argmax(0)\n        polygons, confidence = {}, {}\n        for index, part in enumerate(PARTS, 1):\n            mask = (classes == index).astype("uint8")\n            contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)\n            polygons[part] = []\n            for contour in sorted(contours, key=cv2.contourArea, reverse=True)[:12]:\n                if cv2.contourArea(contour) < 12:\n                    continue\n                points = cv2.approxPolyDP(contour, 2, True).reshape(-1, 2)\n                if len(points) >= 3:\n                    polygons[part].append([[round(int(x) / (SIZE - 1), 4), round(int(y) / (SIZE - 1), 4)]\n                                           for x, y in points])\n            if polygons[part]:\n                confidence[part] = round(float(distribution[index][mask > 0].mean()), 4)\n        return {"source": "model", "polygons": polygons, "partConfidence": confidence}\n\n    if args.image:\n        patches = features(backbone, image_tensor(args.image).unsqueeze(0).to(target))[0]\n        print(json.dumps(result(patches)))\n        return\n    keys = [key for _, _, key in data_rows(args.dataset)]\n    predictions = {key: result(patches)\n                   for key, patches in zip(keys, model_features(backbone, target, args.dataset, keys), strict=True)}\n    (args.dataset / "predictions.json").write_text(json.dumps(predictions) + "\\n")\n    print(f"Predicted {len(predictions)} masks")\n\n\ndef main():\n    parser = argparse.ArgumentParser(description=__doc__)\n    sub = parser.add_subparsers(required=True)\n    fit = sub.add_parser("train")\n    fit.add_argument("dataset", type=Path)\n    fit.add_argument("output", type=Path)\n    fit.add_argument("--weights", type=Path)\n    fit.add_argument("--epochs", type=int, default=18)\n    fit.add_argument("--pseudo", action="store_true")\n    fit.add_argument("--pseudo-threshold", type=float, default=0.92)\n    fit.set_defaults(run=train)\n    apply = sub.add_parser("predict")\n    apply.add_argument("checkpoint", type=Path)\n    source = apply.add_mutually_exclusive_group(required=True)\n    source.add_argument("--dataset", type=Path)\n    source.add_argument("--image", type=Path)\n    apply.add_argument("--weights", type=Path)\n    apply.set_defaults(run=predict)\n    args = parser.parse_args()\n    args.run(args)\n\n\nif __name__ == "__main__":\n    main()\n'
(WORK / "parts.py").write_text(program, encoding="utf-8")


In [ ]:
import sys
sys.path.insert(0, str(WORK))
import torch
from parts import SIZE, device, encoder, features, head, logits

target = device()
backbone = encoder(target, BACKBONE_WEIGHTS)
with torch.inference_mode():
    patches = features(backbone, torch.zeros(1, 3, SIZE, SIZE, device=target))
    probe = logits(head(backbone.num_features).to(target).eval(), patches)
assert probe.shape == (1, 4, SIZE, SIZE) and probe.isfinite().all()
print(f"DINOv3 loaded on {target}, output {tuple(probe.shape)}")
del backbone, patches, probe
if torch.cuda.is_available():
    torch.cuda.empty_cache()


In [ ]:
import shutil
import subprocess
import tempfile

def run(*args):
    subprocess.run([sys.executable, str(WORK / "parts.py"), *map(str, args)], check=True)

if DATASET is not None:
    source = Path(DATASET)
    if not (source / "manifest.json").is_file():
        raise FileNotFoundError(source / "manifest.json")
    weights = ["--weights", BACKBONE_WEIGHTS] if BACKBONE_WEIGHTS else []
    with tempfile.TemporaryDirectory(prefix="takhi-") as temporary:
        dataset = Path(temporary) / "dataset"
        shutil.copytree(source, dataset)
        baseline = WORK / "horse_parts.pt"
        run("train", dataset, baseline, *weights)
        run("predict", baseline, "--dataset", dataset, *weights)
        shutil.copyfile(dataset / "predictions.json", WORK / "predictions.json")
        if RUN_PSEUDO:
            run("train", dataset, WORK / "horse_parts_pseudo.pt", "--pseudo", *weights)
else:
    print("Данные не подключены. Выполнена только проверка модели.")
